# NLP Lab 4 Task

### **Instructions for Students:**
1. All data preparation, vocabularies, and boilerplate loader functions are pre-written.
2. Do **NOT** copy-paste standard models from the lab demo. Read the specific architectural or algorithmic twists for each task carefully.
3. You must replace all `raise NotImplementedError()` blocks with your code implementation.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import gensim.downloader as api
import numpy as np
import random
import urllib.request
import re

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load Pre-trained Google Word2Vec (300-dim)
print("Loading Google Word2Vec Model...")
w2v_google = api.load('word2vec-google-news-300')

def build_embedding_matrix(corpus, w2v_model):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    embedding_dim = w2v_model.vector_size
    weights = [np.zeros(embedding_dim), np.random.randn(embedding_dim) * 0.1]
    
    for sentence in corpus:
        for word in sentence:
            if word not in vocab:
                vocab[word] = len(vocab)
                if word in w2v_model:
                    weights.append(w2v_model[word])
                else:
                    weights.append(np.random.randn(embedding_dim) * 0.1)
                    
    return vocab, torch.tensor(np.array(weights), dtype=torch.float32)

# Seq2Seq with Dot-Product Attention (English to Bengali)
### **The Twist / Challenge:**
Standard Encoder-Decoders struggle because all information must pass through a single fixed context vector. You must upgrade the Decoder by implementing Dot-Product Attention.For each decoder step $t$:
1. Calculate score vector between current decoder hidden state $h_t^d$ and all encoder outputs $H^e$:
$$\text{score}(h_t^d, H^e) = h_t^d \cdot (H^e)^T$$
2. Apply softmax over spatial dimension to get attention weights $\alpha_t$.
3. Compute context vector $c_t = \sum \alpha_t \cdot H^e$.
4. Concatenate $c_t$ and $h_t^d$, then pass through a linear layer before output projection.

In [ ]:
# Data setup (provided)

pairs = [
    ("i love my country", "আমি আমার দেশকে ভালোবাসি"),
    ("how are you", "আপনি কেমন আছেন"),
    ("good morning", "সুপ্রভাত")
]

en_vocab = {"<PAD>": 0, "<UNK>": 1, "<SOS>": 2, "<EOS>": 3}
bn_vocab = {"<PAD>": 0, "<UNK>": 1, "<SOS>": 2, "<EOS>": 3}

for en_s, bn_s in pairs:
    for w in en_s.split():
        if w not in en_vocab: en_vocab[w] = len(en_vocab)
    for w in bn_s.split():
        if w not in bn_vocab: bn_vocab[w] = len(bn_vocab)

en_embed_matrix = torch.randn((len(en_vocab), 300)) # Placeholder pretrained matrix

In [ ]:
class AttentionDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        # Combined context + hidden layer
        self.fc_combined = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        """
        x: (batch, 1) - current decoder input token
        hidden, cell: (1, batch, hidden_dim) - previous decoder state
        encoder_outputs: (batch, seq_len, hidden_dim) - all outputs from encoder
        """
        # TODO 1: Embed input x -> (batch, 1, embed_dim)
        # TODO 2: Run single step of LSTM -> out: (batch, 1, hidden_dim), (hidden, cell)
        # TODO 3: Compute Dot-Product Attention:
        #         a. Query = out (shape: batch, 1, hidden_dim)
        #         b. Keys = encoder_outputs (shape: batch, seq_len, hidden_dim)
        #         c. Compute dot product: torch.bmm(Query, Keys.transpose(1, 2)) -> (batch, 1, seq_len)
        #         d. Apply F.softmax along dim=-1 to get attn_weights
        #         e. Context = torch.bmm(attn_weights, encoder_outputs) -> (batch, 1, hidden_dim)
        # TODO 4: Concatenate Context and out along dim=-1 -> (batch, 1, hidden_dim * 2)
        # TODO 5: Pass through self.fc_combined, apply torch.tanh, then pass through self.fc_out
        # Return: predictions (batch, vocab_size), hidden, cell
        embeds = self.embedding(x)
        out, (hidden, cell) = self.lstm(embeds, (hidden, cell))
        query = out
        keys = encoder_outputs
        scores = torch.bmm(query, keys.transpose(1, 2))
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.bmm(attn_weights, encoder_outputs)
        combined = torch.cat((context, out), dim=-1)
        hidden_proj = torch.tanh(self.fc_combined(combined))
        predictions = self.fc_out(hidden_proj).squeeze(1)
        return predictions, hidden, cell